# Working with NHL Data

This notebook demonstrates how to scrape and analyze NHL data using ScraperNHL.

## NHL vs Other Leagues

The NHL uses a different API than the junior/minor leagues, but ScraperNHL provides a unified interface.

**Note**: Some NHL-specific features may not be available through the unified API yet. For full NHL functionality, you may need to use the legacy NHL scraper modules.

In [ ]:
from scrapernhl import scrape, HockeyScraper
import pandas as pd

# Create an NHL scraper
nhl = HockeyScraper('nhl')

## Getting Play-by-Play Data

The most common use case is getting play-by-play data for a game.

In [ ]:
# Example game ID: 2024020001 (first game of 2024-25 season)
game_id = 2024020001

# Get play-by-play data
pbp = scrape('nhl', 'pbp', game_id=game_id)

print(f"Total events: {len(pbp)}")
print(f"\nColumns: {pbp.columns.tolist()}")
print(f"\nFirst few events:")
display(pbp.head(10))

## Analyzing Play-by-Play Data

Once you have the data, you can perform various analyses.

In [ ]:
# Filter for specific event types
if 'event_type' in pbp.columns:
    goals = pbp[pbp['event_type'] == 'GOAL']
    shots = pbp[pbp['event_type'].isin(['SHOT', 'GOAL'])]
    
    print(f"Goals: {len(goals)}")
    print(f"Total shots (including goals): {len(shots)}")
    
    if len(goals) > 0:
        print("\nGoal details:")
        display(goals[['period', 'time', 'team', 'player', 'event_type']].head())

## Legacy NHL Scraper (Full Features)

For complete NHL functionality, you can use the legacy scraper modules:

In [ ]:
# Import legacy NHL scrapers
from scrapernhl.nhl.scraper import scrapeGame

# This provides more NHL-specific features
game_data = scrapeGame(game_id)

if hasattr(game_data, 'data'):
    print(f"Game events: {len(game_data.data)}")
    print(f"Home team: {game_data.homeTeam if hasattr(game_data, 'homeTeam') else 'N/A'}")
    print(f"Away team: {game_data.awayTeam if hasattr(game_data, 'awayTeam') else 'N/A'}")

## Getting Team Data

You can also get NHL team information:

In [ ]:
# Using legacy scraper for teams
from scrapernhl.nhl.scraper import scrapeTeams

teams = scrapeTeams()
print(f"NHL teams: {len(teams)}")
display(teams.head())

## Getting Schedule Data

Get a team's schedule for the season:

In [ ]:
from scrapernhl.nhl.scraper import scrapeSchedule

# Get Montreal Canadiens schedule for 2024-25 season
schedule = scrapeSchedule("MTL", "20242025")

print(f"Games in schedule: {len(schedule)}")
display(schedule.head(10))

## Player Statistics

Get player stats for a specific player:

In [ ]:
from scrapernhl.nhl.scraper import scrapePlayerSeasonStats

# Connor McDavid's player ID
mcdavid_id = 8478402

# Get his stats for 2024-25 season
stats = scrapePlayerSeasonStats(mcdavid_id, "20242025")

print("Connor McDavid 2024-25 Stats:")
display(stats)

## Standings

Get current NHL standings:

In [ ]:
from scrapernhl.nhl.scraper import scrapeStandings
from datetime import datetime

# Get current standings
standings = scrapeStandings(datetime.now().strftime("%Y-%m-%d"))

print("NHL Standings:")
display(standings.head(15))

## Advanced Analytics

ScraperNHL includes advanced analytics functions:

In [ ]:
# Example: Calculate Corsi from play-by-play data
# Note: This requires the analytics module to be properly set up

try:
    from scrapernhl.nhl.analytics import calculate_corsi, identify_scoring_chances
    
    # Use the pbp data from earlier
    if 'pbp' in locals():
        corsi = calculate_corsi(pbp)
        print("Corsi stats calculated")
        display(corsi.head())
except ImportError:
    print("Analytics module not available in this version")

## Advanced Analytics

ScraperNHL includes advanced analytics functions for Corsi, Fenwick, on-ice stats, and more.

> **Note:** Expected Goals (xG) functionality has been sunset and is no longer part of the public API.

In [ ]:
print("xG functionality has been sunset and is no longer available.")

## Goal Replay Tracking Data

NHL play-by-play goal events include a `pptReplayUrl` field that links to
sprite-based tracking data — real-time (x, y) positions for every skater
and the puck during the goal clip.

Use `goal_replay()` to fetch the raw frames, then `tracking_dict_to_df()`
to convert them into a tidy DataFrame with rink coordinates.

| Column | Description |
|---|---|
| `entity_id` | Sprite index within the frame |
| `playerId` | NHL player ID (NaN for puck) |
| `teamId` | Team ID |
| `x`, `y` | Raw SVG canvas coordinates (pixels) |
| `rink_x`, `rink_y` | Rink coordinates in feet (origin = center ice) |
| `timeStamp` | Frame timestamp |
| `is_puck` | True for the puck entity |
| `frame` | Frame number relative to first frame (`extra=True`) |
| `dx`, `dy`, `dt`, `speed` | Frame-to-frame movement in feet per frame (`extra=True`) |


In [ ]:
from scrapernhl import HockeyScraper, tracking_dict_to_df

nhl = HockeyScraper('nhl')

# Get a game's PBP and extract the pptReplayUrl from a goal event
pbp = nhl.play_by_play(game_id=2023020001)
goal_url = pbp.query("typeDescKey == 'goal'")['pptReplayUrl'].iloc[0]
print("Replay URL:", goal_url)

# Fetch raw sprite frames
frames = nhl.goal_replay(goal_url)
print(f"{len(frames)} frames fetched")

# Convert to tidy DataFrame
tracking_df = tracking_dict_to_df(frames)
print(f"Shape: {tracking_df.shape}")
print(f"Columns: {list(tracking_df.columns)}")
tracking_df.head(10)


## Exporting Data

Export your data to various formats:

In [ ]:
# Export play-by-play to CSV
if 'pbp' in locals() and len(pbp) > 0:
    pbp.to_csv('nhl_pbp.csv', index=False)
    print("Exported to nhl_pbp.csv")

# Export to JSON
if 'pbp' in locals() and len(pbp) > 0:
    pbp.to_json('nhl_pbp.json', orient='records', indent=2)
    print("Exported to nhl_pbp.json")

# Export to Excel (requires openpyxl)
try:
    if 'pbp' in locals() and len(pbp) > 0:
        pbp.to_excel('nhl_pbp.xlsx', index=False)
        print("Exported to nhl_pbp.xlsx")
except ImportError:
    print("Install openpyxl to export to Excel: pip install openpyxl")

## Summary

This notebook covered:
- Getting NHL play-by-play data
- Using legacy NHL scraper functions
- Accessing teams, schedules, and standings
- Player statistics
- Advanced analytics (Corsi, Fenwick, on-ice stats)
- Goal replay tracking data with `tracking_dict_to_df()`
- Exporting data

## Next Steps

- **03_other_leagues.ipynb** - Learn about QMJHL, OHL, WHL, and AHL
- **04_advanced_features.ipynb** - Caching, rate limiting, and transformations